In [1]:
#!pip install xgboost

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('credit_customers_train.csv')

In [3]:
#to check missing colums
df.isnull().values.any()

False

In [4]:

#To check duplicates
df[df.duplicated()]

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class


In [5]:
# Map target: good=0, bad=1
df['class'] = df['class'].map({'good': 0, 'bad': 1})

In [6]:
# Features and target

X = df.drop('class', axis=1)
y = df['class']

correlation = df.corr(numeric_only=True)['class'].sort_values(ascending=False)
print(correlation)

# Define columns
num_cols = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
cat_cols = ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status',
            'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']

class                     1.000000
duration                  0.219545
credit_amount             0.161800
installment_commitment    0.040151
residence_since          -0.007398
num_dependents           -0.025392
existing_credits         -0.065230
age                      -0.116145
Name: class, dtype: float64


<h>Observation: None of the numeric features have strong linear correlation (> 0.5).</h>

In [7]:
#To check class imbalance ratio
df['class'].value_counts(normalize=True) * 100

class
0    70.0
1    30.0
Name: proportion, dtype: float64

<h>Observation : Minority class is less than < 40% > 20% moderate imbalance is observed in the dataset. Class weights must be used</h>

In [8]:
df.groupby('class')[num_cols].mean()

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents
class,,,,,,,
0,19.557143,2960.214286,2.948571,2.811429,35.885714,1.425714,1.16
1,25.493333,3964.900000,3.046667,2.793333,33.133333,1.346667,1.14


<h> Observation: Class 1 customers borrow more money Class 1 Loans have longer loan tenure</h>

In [9]:
df[num_cols].skew()

duration                  1.127215
credit_amount             1.942473
installment_commitment   -0.556919
residence_since          -0.236107
age                       1.002010
existing_credits          1.130959
num_dependents            1.922943
dtype: float64

Since  credit_amount skewness = 1.9424 (very high):
Large loan amounts stretch the distribution
Model may get influenced by extreme values

In [10]:
df['credit_amount_log'] = np.log1p(df['credit_amount'])
df['duration_log'] = np.log1p(df['duration'])

In [11]:
df.drop('credit_amount', axis=1, inplace=True)
df.drop('duration', axis=1, inplace=True)

In [13]:
df['housing'].value_counts()

housing
own         356
rent         90
for free     54
Name: count, dtype: int64

In [12]:
from scipy.stats import chi2_contingency

results = []
for col in cat_cols:
    table = pd.crosstab(df[col], df['class'])
    chi2, p, dof, expected = chi2_contingency(table)
    
    results.append({
        'Feature': col,
        'Chi2': chi2,
        'P-value': p,
        'Significant (<0.05)': p < 0.05
    })

chi_square_results = pd.DataFrame(results)
chi_square_results.sort_values(by='P-value')

,Feature,Chi2,P-value,Significant (<0.05)
0,checking_status,56.468066,3.337746e-12,True
1,credit_history,36.035103,2.845978e-07,True
7,property_magnitude,15.162377,1.683015e-03,True
3,savings_status,14.119545,6.923132e-03,True
9,housing,9.217546,9.964039e-03,True
4,employment,10.856811,2.822128e-02,True
2,purpose,18.094320,3.409490e-02,True
8,other_payment_plans,5.707338,5.763249e-02,False
6,other_parties,3.466981,1.766666e-01,False
5,personal_status,3.651718,3.016113e-01,False


<h>From above chi square test its observed that the following features have significant relationship with target variable
checking_status
credit_history
property_magnitude
savings_status
housing
employment
purpose

checking_status and credit_history have low p-values, indicating a very strong association with credit risk.

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'), cat_cols)
    ])

# Split (stratify due to imbalance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit preprocessor on train
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

# Save preprocessor
os.makedirs('model', exist_ok=True)
joblib.dump(preprocessor, 'model/preprocessor.joblib')

['model/preprocessor.joblib']

In [14]:
def train_evaluate_save(model, name):
    model.fit(X_train_pre, y_train)
    y_pred = model.predict(X_test_pre)
    y_prob = model.predict_proba(X_test_pre)[:, 1]  
    
   
    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_prob),
        "mcc": matthews_corrcoef(y_test, y_pred)
    }
    
      
    joblib.dump(model, f'model/{name.lower().replace(" ", "_")}.joblib')
    return results

In [15]:
import json

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5, class_weight='balanced'),
    'kNN': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=700/300)
}

results = {}

for name, model in models.items():
    results[name] = train_evaluate_save(model, name)

# Save to JSON
with open("model_metrics.json", "w") as f:
    json.dump(results, f, indent=4)

df = pd.DataFrame(results).T 
pd.set_option("display.width", 2000)
pd.set_option("display.max_columns", None)
print(df)


                     accuracy  precision    recall        f1       auc       mcc
Logistic Regression      0.64   0.437500  0.700000  0.538462  0.739048  0.288278
Decision Tree            0.61   0.390244  0.533333  0.450704  0.671905  0.164163
kNN                      0.70   0.500000  0.366667  0.423077  0.635714  0.231784
Naive Bayes              0.62   0.404762  0.566667  0.472222  0.676667  0.194538
Random Forest            0.68   0.400000  0.133333  0.200000  0.732381  0.072739
XGBoost                  0.69   0.481481  0.433333  0.456140  0.686667  0.240848
